# Fine-tune YOLO11n — **2-class** (`datamatrix` + `CODE128`) — Kaggle

Locate **Data Matrix** + **Code 128**. GS1-128 labels are remapped into CODE128 (same symbology family). EAN-13 dropped.

ZXing still decodes; YOLO only draws boxes.

## Classes
- `0: datamatrix`
- `1: CODE128`

## Setup
1. Dataset input with:
   - `best.pt` (qr_det or datamatrix run)
   - `barcode_best_training_images/` or `.zip`
   - `barcode_dataset_merged/` or `.zip` (source classes: 0=EAN13, 1=CODE128, 2=GS1 — **1** CODE128 + **2** GS1 → class 1)
2. Session: **GPU T4**, Internet **On**
3. Run top → bottom; download `best_dm_code128.pt` + `barcode-yolo11n-dm-code128.onnx`


## 1) GPU check + install


In [ ]:
!nvidia-smi
!pip -q install -U ultralytics

import torch
import ultralytics
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("ultralytics", ultralytics.__version__)
assert torch.cuda.is_available(), "Enable GPU: Session options → Accelerator → GPU T4, then Restart session"


## 2) Stage dataset (folders or zips)


In [ ]:
from pathlib import Path
import shutil
import zipfile

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

print("Inputs:")
!ls -la /kaggle/input

hits = list(INPUT_ROOT.rglob("best.pt"))
assert hits, "best.pt not found under /kaggle/input — Add Input dataset first"
DATASET_DIR = hits[0].parent
print("Using dataset dir:", DATASET_DIR)
!ls -lah {DATASET_DIR}

shutil.copy2(DATASET_DIR / "best.pt", WORK / "best.pt")
print("Copied best.pt")


def find_asset(name_stem: str):
    z = DATASET_DIR / f"{name_stem}.zip"
    d = DATASET_DIR / name_stem
    if z.exists():
        return "zip", z
    if d.exists() and d.is_dir():
        return "dir", d
    zips = list(INPUT_ROOT.rglob(f"{name_stem}.zip"))
    if zips:
        return "zip", zips[0]
    dirs = [p for p in INPUT_ROOT.rglob(name_stem) if p.is_dir()]
    if dirs:
        return "dir", dirs[0]
    return None, None


def stage_dataset(name_stem: str, dest: Path):
    kind, src = find_asset(name_stem)
    assert src is not None, f"Missing {name_stem} (.zip or folder) under /kaggle/input"
    print(f"{name_stem}: {kind} -> {src}")
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    if kind == "zip":
        with zipfile.ZipFile(src, "r") as zf:
            zf.extractall(dest)
    else:
        shutil.copytree(src, dest, dirs_exist_ok=True)
    return dest


def unwrap_if_nested(root: Path) -> Path:
    if (root / "images").exists():
        return root
    nested = list(root.rglob("images/train"))
    if nested:
        return nested[0].parent.parent
    for p in root.rglob("images"):
        if (p / "train").exists():
            return p.parent
    return root


dm_root = unwrap_if_nested(stage_dataset("barcode_best_training_images", WORK / "dm_ds"))
bc_root = unwrap_if_nested(stage_dataset("barcode_dataset_merged", WORK / "bc_ds"))

print("DM root:", dm_root, "images?", (dm_root / "images").exists())
print("BC root:", bc_root, "images?", (bc_root / "images").exists())
assert (dm_root / "images").exists(), f"DM images not found under {dm_root}"
assert (bc_root / "images").exists(), f"BC images not found under {bc_root}"

Path("/kaggle/working/DM_ROOT.txt").write_text(str(dm_root))
Path("/kaggle/working/BC_ROOT.txt").write_text(str(bc_root))
print("OK — staged datasets")


## 3) Build combined **2-class** dataset

- DataMatrix → class `0`
- Merged barcode: keep source `1` CODE128 + `2` GS1_128 → class `1`
- Drop EAN13 (`0`); skip images with no remaining boxes


In [ ]:
import shutil
from collections import Counter
from pathlib import Path

dm_root = Path(Path("/kaggle/working/DM_ROOT.txt").read_text().strip())
bc_root = Path(Path("/kaggle/working/BC_ROOT.txt").read_text().strip())

combined = Path("/kaggle/working/combined_2class_dm_code128")
if combined.exists():
    shutil.rmtree(combined)

for split in ["train", "val", "test"]:
    (combined / "images" / split).mkdir(parents=True, exist_ok=True)
    (combined / "labels" / split).mkdir(parents=True, exist_ok=True)

valid_ext = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
NL = chr(10)


def remap_lines(src_lbl, class_map):
    out = []
    if not src_lbl.exists():
        return out
    for line in src_lbl.read_text().strip().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        src_cid = int(float(parts[0]))
        if src_cid not in class_map:
            continue
        out.append(" ".join([str(class_map[src_cid])] + parts[1:]))
    return out


def copy_pair(src_img, src_lbl, dst_img_dir, dst_lbl_dir, class_map, out_stem):
    if not src_img.exists():
        return False
    lines = remap_lines(src_lbl, class_map)
    if not lines:
        return False
    shutil.copy2(src_img, dst_img_dir / f"{out_stem}{src_img.suffix.lower()}")
    (dst_lbl_dir / f"{out_stem}.txt").write_text(NL.join(lines) + NL)
    return True


dm_map = {i: 0 for i in range(32)}
bc_map = {1: 1, 2: 1}  # CODE128 + GS1_128 → CODE128

dm_counts = Counter()
for split in ["train", "val", "test"]:
    dm_img = dm_root / "images" / split
    dm_lbl = dm_root / "labels" / split
    if not dm_img.exists() or not dm_lbl.exists():
        continue
    for img in sorted(dm_img.iterdir()):
        if img.suffix.lower() not in valid_ext:
            continue
        if copy_pair(img, dm_lbl / f"{img.stem}.txt", combined / "images" / split, combined / "labels" / split, dm_map, f"dm_{img.stem}"):
            dm_counts[split] += 1

bc_counts = Counter()
skipped_no_code128 = 0
for split in ["train", "val", "test"]:
    bc_img = bc_root / "images" / split
    bc_lbl = bc_root / "labels" / split
    if not bc_img.exists() or not bc_lbl.exists():
        continue
    for img in sorted(bc_img.iterdir()):
        if img.suffix.lower() not in valid_ext:
            continue
        ok = copy_pair(img, bc_lbl / f"{img.stem}.txt", combined / "images" / split, combined / "labels" / split, bc_map, f"bc_{img.stem}")
        if ok:
            bc_counts[split] += 1
        else:
            skipped_no_code128 += 1

print("DM image counts:", dict(dm_counts))
print("CODE128 image counts:", dict(bc_counts))
print("Skipped BC images (no CODE128 box):", skipped_no_code128)
Path("/kaggle/working/COMBINED_ROOT.txt").write_text(str(combined))
print("Combined:", combined)


## 4) Write `data.yaml` + class counts


In [ ]:
from pathlib import Path
from collections import Counter

combined = Path(Path("/kaggle/working/COMBINED_ROOT.txt").read_text().strip())

yaml_text = """path: /kaggle/working/combined_2class_dm_code128
train: images/train
val: images/val
test: images/test
nc: 2
names:
  0: datamatrix
  1: CODE128
"""
(combined / "data.yaml").write_text(yaml_text)

img_counts = {}
box_counts = Counter()
for split in ["train", "val", "test"]:
    imgs = list((combined / "images" / split).glob("*"))
    img_counts[split] = len([p for p in imgs if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}])
    for lf in (combined / "labels" / split).glob("*.txt"):
        for line in lf.read_text().strip().splitlines():
            if line.strip():
                box_counts[int(float(line.split()[0]))] += 1

print("Image counts:", img_counts)
print("Box counts by class:", dict(sorted(box_counts.items())))
assert sum(img_counts.values()) > 0
assert img_counts.get("val", 0) > 0, "val split empty"
assert box_counts.get(0, 0) > 0 and box_counts.get(1, 0) > 0, "Need both datamatrix and CODE128 boxes"
print("OK data.yaml")


## 5) Fine-tune from `best.pt`


In [ ]:
from ultralytics import YOLO
from pathlib import Path

BEST_PT = Path("/kaggle/working/best.pt")
DATA_YAML = Path("/kaggle/working/combined_2class_dm_code128/data.yaml")
assert BEST_PT.exists() and DATA_YAML.exists()

PROJECT = "/kaggle/working/runs/barcode"
RUN_NAME = "yolo11n_960_dm_code128_v1"
IMGSZ = 960
EPOCHS = 120
BATCH = 8  # drop to 4 if OOM

model = YOLO(str(BEST_PT))
results = model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    patience=35,
    freeze=10,
    cls=1.0,
    box=7.5,
    close_mosaic=15,
    cos_lr=True,
)
print("Done:", Path(PROJECT) / RUN_NAME / "weights")


## 6) Validate


In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = Path("/kaggle/working/runs/barcode")
candidates = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert candidates, "No best.pt"
BEST = candidates[0]
print("Using", BEST)

model = YOLO(str(BEST))
metrics = model.val(data="/kaggle/working/combined_2class_dm_code128/data.yaml", imgsz=960)
print("mAP50:", float(metrics.box.map50))
print("mAP50-95:", float(metrics.box.map))
print("Precision:", float(metrics.box.mp))
print("Recall:", float(metrics.box.mr))
try:
    print("Per-class mAP50:", metrics.box.maps)
except Exception as e:
    print("per-class:", e)


## 7) Export ONNX (`1 × 6 × N`)


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import shutil

PROJECT = Path("/kaggle/working/runs/barcode")
candidates = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
BEST = candidates[0]
print("Exporting", BEST)

model = YOLO(str(BEST))
model.export(format="onnx", imgsz=960, simplify=True)

onnx_src = BEST.with_name("best.onnx")
if not onnx_src.exists():
    onnx_src = BEST.parent / "best.onnx"
assert onnx_src.exists(), f"ONNX missing near {BEST}"

onnx_dst = Path("/kaggle/working/barcode-yolo11n-dm-code128.onnx")
shutil.copy2(onnx_src, onnx_dst)
shutil.copy2(BEST, "/kaggle/working/best_dm_code128.pt")
print("Wrote", onnx_dst, "size", onnx_dst.stat().st_size)
print("Wrote /kaggle/working/best_dm_code128.pt")
print("Expected ONNX output shape: 1 x 6 x N  (4 box + 2 class)")


## 8) Download

```bash
mkdir -p runs/barcode/yolo11n_960_dm_code128_v1/weights
cp ~/Downloads/best_dm_code128.pt runs/barcode/yolo11n_960_dm_code128_v1/weights/best.pt
cp ~/Downloads/barcode-yolo11n-dm-code128.onnx public/models/barcode-yolo11n.onnx
python3 scripts/probe-yolo-classes.py
```

App class names: `datamatrix`, `CODE128`.


In [ ]:
from IPython.display import FileLink, display
from pathlib import Path

for p in [
    Path("/kaggle/working/best_dm_code128.pt"),
    Path("/kaggle/working/barcode-yolo11n-dm-code128.onnx"),
]:
    print(p, "exists" if p.exists() else "MISSING", p.stat().st_size if p.exists() else "")
    if p.exists():
        display(FileLink(str(p)))


## Notes
- **OOM:** `BATCH=4`
- Keep `imgsz=960` for small DataMatrix on pallets
- GS1-128 labels are remapped to CODE128 class — retrain after updating dataset
- App also proposes CODE128 strips beside detected DataMatrix on pharma labels
